# 02 — Normalize listings

Map messy titles (`2021 Camry SE`, `Toyota Camry Sport`) to canonical make/model using
`vehicle_make` / `vehicle_model` from Postgres (fallback catalog if DB unreachable).

Run after notebook 01 or standalone with the bundled sample feed.

In [ ]:
import pandas as pd

from notification_rake.config import settings
from notification_rake.ingestion import fetch_listings
from notification_rake.transform import load_catalog, normalize_listings

raw = fetch_listings(settings.craigslist_search_rss)
catalog = load_catalog(settings.database_url)
normalized = normalize_listings(raw, catalog)

df = pd.DataFrame([n.model_dump() for n in normalized])
print(f"Catalog entries: {len(catalog)} | Listings: {len(df)}")
df[["title", "make", "model", "year", "price"]]

In [ ]:
unmatched = df[df["make"].isna()]
if len(unmatched):
    print(f"Unmatched ({len(unmatched)}):")
    display(unmatched[["title"]])
else:
    print("All listings matched a canonical make/model.")

In [ ]:
# Example: register a new model (no-op if already in catalog)
from notification_rake.storage import add_vehicle_model

try:
    model_id = add_vehicle_model(settings.database_url, "Toyota", "Supra")
    print(f"Toyota Supra -> model_id={model_id}")
except Exception as exc:
    print(f"Skipped ({exc}) — run `docker compose up -d db` first")

## See also

Matching logic lives in `notification_rake.normalize`. Next: `03_geospatial_search.ipynb`.